In [3]:
from dotenv import load_dotenv
from langchain_community.document_loaders  import PyPDFLoader

load_dotenv()

file_path="data/한글맞춤법 표준어규정 해설.pdf"

loader = PyPDFLoader(file_path)
pages = []

async for page in loader.alazy_load():
    pages.append(page)

In [4]:
print("페이지 수: ",len(pages))
print("첫번째 페이지 정보: ",pages[0])

페이지 수:  264
첫번째 페이지 정보:  page_content='국립국어원 2018-01-08
발간 등록 번호
11-1371028-000712-01
한글 맞춤법
표준어 규정
해설' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2018-12-20T09:36:29+09:00', 'author': 'admin', 'moddate': '2018-12-20T09:38:09+09:00', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'source': 'data/한글맞춤법 표준어규정 해설.pdf', 'total_pages': 264, 'page': 0, 'page_label': '1'}


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
docs = text_splitter.split_documents(pages)

In [ ]:
print(f"총 {len(docs)}개의 청크 생성 완료")
print("각 청크의 길이: ",  *[f'[ {i} ] = {len(doc.page_content)} \n' for i, doc in enumerate(docs)])

# for i in docs:
#     print(f"[메타데이터] {i.metadata}")
#     print(f"[내용] {i.page_content}")
#     print("=" * 50)

총 510개의 청크 생성 완료
각 청크의 길이:  [ 0 ] = 63 
 [ 1 ] = 467 
 [ 2 ] = 367 
 [ 3 ] = 472 
 [ 4 ] = 96 
 [ 5 ] = 473 
 [ 6 ] = 110 
 [ 7 ] = 343 
 [ 8 ] = 458 
 [ 9 ] = 471 
 [ 10 ] = 499 
 [ 11 ] = 129 
 [ 12 ] = 484 
 [ 13 ] = 456 
 [ 14 ] = 461 
 [ 15 ] = 124 
 [ 16 ] = 11 
 [ 17 ] = 420 
 [ 18 ] = 448 
 [ 19 ] = 448 
 [ 20 ] = 466 
 [ 21 ] = 463 
 [ 22 ] = 220 
 [ 23 ] = 489 
 [ 24 ] = 486 
 [ 25 ] = 484 
 [ 26 ] = 466 
 [ 27 ] = 54 
 [ 28 ] = 454 
 [ 29 ] = 476 
 [ 30 ] = 288 
 [ 31 ] = 468 
 [ 32 ] = 167 
 [ 33 ] = 468 
 [ 34 ] = 475 
 [ 35 ] = 452 
 [ 36 ] = 79 
 [ 37 ] = 452 
 [ 38 ] = 217 
 [ 39 ] = 474 
 [ 40 ] = 494 
 [ 41 ] = 268 
 [ 42 ] = 476 
 [ 43 ] = 332 
 [ 44 ] = 480 
 [ 45 ] = 242 
 [ 46 ] = 450 
 [ 47 ] = 451 
 [ 48 ] = 450 
 [ 49 ] = 202 
 [ 50 ] = 497 
 [ 51 ] = 259 
 [ 52 ] = 486 
 [ 53 ] = 482 
 [ 54 ] = 119 
 [ 55 ] = 491 
 [ 56 ] = 124 
 [ 57 ] = 458 
 [ 58 ] = 429 
 [ 59 ] = 456 
 [ 60 ] = 415 
 [ 61 ] = 492 
 [ 62 ] = 106 
 [ 63 ] = 479 
 [ 64 ] = 250 
 [ 65 ] = 489

In [20]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
DB_PATH = Path.cwd() / "database" / "chroma_db"
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="korean_pdf"
)

In [22]:
print(DB_PATH)

d:\agent\src\single_agent\rag_agent\database\chroma_db


In [21]:

vectorstore.similarity_search("구개음화", k=3)

[Document(id='b0cc72c3-38ae-400f-8b36-0ade195b4af0', metadata={'moddate': '2018-12-20T09:38:09+09:00', 'source': 'data/한글맞춤법 표준어규정 해설.pdf', 'page': 23, 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creationdate': '2018-12-20T09:36:29+09:00', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'page_label': '24', 'author': 'admin', 'total_pages': 264}, page_content='는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이\n어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 말이므로 구개음화가 실현되지 \n않는다. \n곧이[고지]: 곧-(어근)+-이(부사 파생 접미사)\n곧이어[고디어]: 곧(부사)+이어(부사)\n현재 표준어에서 구개음화는 형태소와 형태소가 결합할 때 일어나는 현상이다. 그러\n므로 ‘마디, 견디다’와 같이 하나의 형태소 내부에서는 구개음화가 일어나지 않는다.'),
 Document(id='45445aab-87e9-486c-9544-63b87782cf0c', metadata={'creationdate': '2018-12-20T09:36:29+09:00', 'total_pages': 264, 'page': 23, 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '24', 'moddate': '2018-12-20T09:38:09+09:00', 'author': 'admin', 'creator': 'PScrip